# 🎬 NARRATIVE NEXUS — Step 4: Full Backend on Colab

**Run cells 1 → 5 in order. After Cell 4, copy the ngrok URL into your frontend.**

Keep this notebook running the whole time you're using the app.

## 🔧 Cell 1: Install Everything

In [1]:
!pip install -q flask flask-cors pyngrok groq
!pip install -q diffusers transformers accelerate
!pip install -q opencv-python moviepy Pillow
!apt-get install -qq ffmpeg
!pip install -q gtts
print('✅ All installed!')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
✅ All installed!


## 🔑 Cell 2: Set API Keys

- **Groq key** (free): https://console.groq.com
- **ngrok token** (free): https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
import os

os.environ['GROQ_API_KEY'] = 'ADD_YOUR_GROQ_API_KEY_HERE'
NGROK_TOKEN               = 'ADD_YOUR_NGROK_TOKEN_HERE'

groq_set  = os.environ['GROQ_API_KEY'] not in ('', 'YOUR_GROQ_KEY_HERE')
ngrok_set = NGROK_TOKEN not in ('', 'YOUR_NGROK_TOKEN_HERE')

print('GROQ :', '✅' if groq_set  else '⚠️  not set')
print('ngrok:', '✅' if ngrok_set else '❌ REQUIRED')

GROQ : ✅
ngrok: ✅


## 📁 Cell 3: Mount Drive + Setup Demo Folder

This mounts your Google Drive and copies demo files automatically every session.

**One-time setup:** Upload your demo files to Google Drive at `My Drive/NarrativeNexus/demo/`:
- `a girl entered the park.mp4`
- `a girl entered the park.txt`

After that, this cell handles everything automatically on every session restart.

In [3]:
from google.colab import drive
import os, shutil

# Mount Google Drive
drive.mount('/content/drive')

# ── Create demo folder ──────────────────────────────────────
DEMO_DIR   = '/content/demo'
DRIVE_DIR  = '/content/drive/MyDrive/NarrativeNexus/demo'

os.makedirs(DEMO_DIR, exist_ok=True)

# ── Copy demo files from Drive if they exist ────────────────
if os.path.isdir(DRIVE_DIR):
    copied = 0
    for fname in os.listdir(DRIVE_DIR):
        src = os.path.join(DRIVE_DIR, fname)
        dst = os.path.join(DEMO_DIR, fname)
        shutil.copy2(src, dst)
        print(f'  ✅ Copied: {fname}')
        copied += 1
    print(f'\n📁 Demo folder ready with {copied} file(s)!')
else:
    print(f'⚠️  Drive folder not found at: {DRIVE_DIR}')
    print('   Create this folder in Google Drive and upload your demo files there.')
    print('   Or upload directly to /content/demo/ via the Files panel.')

# ── Confirm app.py exists ───────────────────────────────────
assert os.path.exists('app.py'), '❌ app.py not found! Upload it to the Files panel first.'
print('\n✅ app.py found and ready!')

# ── Show what's in demo folder ──────────────────────────────
print(f'\n📂 Contents of {DEMO_DIR}:')
for f in os.listdir(DEMO_DIR):
    size = os.path.getsize(os.path.join(DEMO_DIR, f))
    print(f'   {f}  ({size/1024/1024:.1f} MB)')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  ✅ Copied: a girl entered the park.mp4
  ✅ Copied: a girl entered the park.txt

📁 Demo folder ready with 2 file(s)!

✅ app.py found and ready!

📂 Contents of /content/demo:
   a girl entered the park (4).mp4  (35.2 MB)
   a girl entered the park.mp4  (35.2 MB)
   a girl entered the park.txt  (0.0 MB)


## 🚀 Cell 4: Start Flask + Get Public URL

**After running this cell:**
1. Copy the `🌐 Public URL` printed below
2. Paste it into the frontend app when prompted
3. Hit Connect

> ⚠️ This URL changes every time you restart. Paste the new one each session.

In [4]:
import sys, time, threading
from pyngrok import ngrok

# Reload app module fresh each run
for mod in list(sys.modules.keys()):
    if mod == 'app': del sys.modules[mod]

from app import app as flask_app

# Start Flask in background thread
def run():
    flask_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

ngrok.set_auth_token(NGROK_TOKEN)
t = threading.Thread(target=run, daemon=True)
t.start()
time.sleep(2)

# Kill any existing tunnels and open a fresh one
ngrok.kill()
time.sleep(1)
tunnel     = ngrok.connect(5000)
public_url = tunnel.public_url

print('=' * 60)
print('🎬  NARRATIVE NEXUS BACKEND IS LIVE')
print('=' * 60)
print(f'\n🌐  Public URL  : {public_url}')
print(f'\n🔍  Health check: {public_url}/health')
print('\n⚠️   Keep this notebook running while using the app!')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


 * Serving Flask app 'app'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://172.28.0.12:5000
INFO:werkzeug:Press CTRL+C to quit


🎬  NARRATIVE NEXUS BACKEND IS LIVE

🌐  Public URL  : https://02dc-34-186-20-241.ngrok-free.app

🔍  Health check: https://02dc-34-186-20-241.ngrok-free.app/health

⚠️   Keep this notebook running while using the app!


## 🧪 Cell 5: Quick Health Check (Optional)

In [5]:
import requests

r = requests.get(f'{public_url}/health',
                 headers={'ngrok-skip-browser-warning': 'true'})
print('Status:', r.status_code)
print('Response:', r.json())

# Also verify demo folder
import os
print('\n📂 Demo folder contents:')
for f in os.listdir('/content/demo'):
    size = os.path.getsize(f'/content/demo/{f}')
    print(f'  {f}  ({size/1024/1024:.1f} MB)')

INFO:werkzeug:127.0.0.1 - - [27/Mar/2026 09:19:12] "GET /health HTTP/1.1" 200 -


Status: 200
Response: {'demo_files': ['a girl entered the park (4).mp4', 'a girl entered the park.mp4', 'a girl entered the park.txt'], 'demo_folder': True, 'gpu': True, 'status': 'ok'}

📂 Demo folder contents:
  a girl entered the park (4).mp4  (35.2 MB)
  a girl entered the park.mp4  (35.2 MB)
  a girl entered the park.txt  (0.0 MB)
